# 03 — Anomaly Detection & Data Cleaning

**Phase 2, Week 4** — CMU-Africa educational tutorial demonstrating how unsupervised AI can support energy efficiency.

**Goal:** Use Isolation Forest and DBSCAN to flag suspicious consumption patterns in smart meter data, then clean the series so Phase 3 forecasting models see a continuous 30-minute timeline.

**Scope (this notebook):** Sections 1–2 load data and build features; Sections 3–4 train detectors and benchmark them against labels; Section 5 (next) covers interpolation and clean data for Phase 3.

**Important:** The dataset includes an `Anomaly_Label` column, but our detectors are **unsupervised** — they never use labels during training. Labels are for benchmark comparison only.

## 1. Setup & Data Loading

**Why:** Every analysis in this project starts from the validated ingestion pipeline established in Phase 1 Week 1. Reusing `load_smart_meter_data` keeps the notebook reproducible and aligned with `src/` — the same path our scripts and docs use.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.ingest_data import find_dataset_csv, get_project_root, load_smart_meter_data
from src.features.build_features import build_all_features

csv_path = find_dataset_csv(get_project_root())
df_raw = load_smart_meter_data(csv_path)

print(f"Loaded: {csv_path}")
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")

## 2. Feature Engineering

**Why:** Raw consumption alone is not enough — a high reading at 2 PM may be normal while the same value at 3 AM may not. Anomaly detectors need **temporal context** (hour, day-of-week) and **rolling memory** (recent local averages). `build_all_features` applies the same pipeline used by our training scripts in `src/features/`.

In [ ]:
df = build_all_features(df_raw)

print(f"Shape after feature engineering: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}")

display(df.head())
display(df.tail())

## 3. Two Unsupervised Detectors

We compare two classic anomaly detectors that learn **only from feature patterns** — never from `Anomaly_Label`.

**Isolation Forest**

- Builds many random trees over the feature matrix.
- Points that are **easy to isolate** (few splits needed) are flagged as anomalies.
- Works well on high-dimensional tabular data; our default assumes ~5% contamination (`contamination=0.05`).

**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**

- Groups dense neighborhoods into clusters; points in **sparse regions** are labeled noise (anomalies).
- Sensitive to hyperparameters: `eps` (neighborhood radius) and `min_samples` (minimum cluster size).
- We use the grid-search winners from Week 4 Day 2: `eps=0.5`, `min_samples=5`.

**Why both?** Different algorithms make different assumptions. Comparing them on the same benchmark helps students see trade-offs before we pick a model for data cleaning.

## 4. Train, Predict, and Benchmark

**Why:** In production we only have meter readings — no ground-truth anomaly labels. The Kaggle dataset includes `Anomaly_Label` so we can **measure** how well unsupervised detectors align with human-annotated spikes and drops.

**Evaluation rules (Phase 2 strategy):**

- Labels are mapped to `0` = Normal, `1` = Abnormal.
- We report **precision, recall, and F1** on the Abnormal class — not accuracy (the dataset is imbalanced).
- Rolling features drop the first 47 rows (warm-up window), so both models and labels are evaluated on **4,953 rows**.

We train via `detect_anomalies` — the same unified router used by our scripts — then score predictions with `evaluate_anomaly_model`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from src.models.evaluate_models import evaluate_anomaly_model
from src.models.train_anomaly_models import detect_anomalies, prepare_feature_matrix

sns.set_theme(style="whitegrid")

clean_index = prepare_feature_matrix(df).index
y_true = (
    df.loc[clean_index, "Anomaly_Label"]
    .map({"Normal": 0, "Abnormal": 1})
    .to_numpy(dtype=int)
)

print(f"Benchmark labels aligned: {len(y_true)} rows")
print(f"  Normal:   {(y_true == 0).sum()}")
print(f"  Abnormal: {(y_true == 1).sum()}")

In [ ]:
_, if_preds = detect_anomalies(df, model_type="isolation_forest")
_, db_preds = detect_anomalies(df, model_type="dbscan", eps=0.5, min_samples=5)

if_metrics = evaluate_anomaly_model(y_true, if_preds)
db_metrics = evaluate_anomaly_model(y_true, db_preds)

comparison = pd.DataFrame(
    [
        {
            "Model": "Isolation Forest",
            "Precision": if_metrics["precision"],
            "Recall": if_metrics["recall"],
            "F1": if_metrics["f1"],
            "Pred. anomalies": int(if_preds.sum()),
        },
        {
            "Model": "DBSCAN (eps=0.5, min_samples=5)",
            "Precision": db_metrics["precision"],
            "Recall": db_metrics["recall"],
            "F1": db_metrics["f1"],
            "Pred. anomalies": int(db_preds.sum()),
        },
    ]
)

display(comparison.round(3))

print(
    "\nTakeaway: Isolation Forest is the stronger baseline on this dataset "
    f"(F1 {if_metrics['f1']:.3f} vs {db_metrics['f1']:.3f})."
)

In [ ]:
def plot_confusion_matrix(ax, cm, title: str) -> None:
    """Plot a 2x2 confusion matrix with TN/FP/FN/TP annotations."""
    labels = np.array([["TN", "FP"], ["FN", "TP"]])
    sns.heatmap(
        cm,
        annot=labels,
        fmt="",
        cmap="Blues",
        cbar=False,
        xticklabels=["Normal", "Abnormal"],
        yticklabels=["Normal", "Abnormal"],
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")


fig, axes = plt.subplots(1, 2, figsize=(10, 4))

plot_confusion_matrix(axes[0], if_metrics["confusion_matrix"], "Isolation Forest")
plot_confusion_matrix(
    axes[1],
    db_metrics["confusion_matrix"],
    "DBSCAN (eps=0.5, min_samples=5)",
)

plt.tight_layout()
plt.show()